In [1]:
import sys, os, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import nltk
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.append("../src")
from model import SentimentNet
from utils import get_attention_map, plot_attention_heatmap

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [2]:
# LSTM
with open("../models/lstm_sentiment/vocab.json") as f:
    word2idx = json.load(f)
lstm_model = SentimentNet(len(word2idx), 2, 128, 128, 2)
lstm_model.load_state_dict(torch.load("../models/lstm_sentiment/model.pt", map_location=device))
lstm_model.to(device).eval()

# DistilBERT
bert_path = "../models/distilbert_sentiment"
tokenizer = AutoTokenizer.from_pretrained(bert_path)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_path, output_attentions=True, attn_implementation="eager")
bert_model.to(device).eval()
print("Both models loaded.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Both models loaded.


In [3]:
def predict_lstm(text, max_len=200):
    tokens = nltk.word_tokenize(str(text).lower())
    encoded = [word2idx.get(w, 1) for w in tokens][:max_len]   # 1 = unknown word
    features = np.zeros((1, max_len), dtype=int)
    if encoded:
        features[0, -len(encoded):] = encoded                  # pre-padding
    with torch.no_grad():
        probs = F.softmax(lstm_model(torch.from_numpy(features).to(device)), dim=1)
    pos = probs[0][1].item()
    return ("POSITIVE" if pos >= 0.5 else "NEGATIVE"), pos


def predict_bert(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        probs = F.softmax(bert_model(**inputs).logits, dim=1)
    pos = probs[0][1].item()
    return ("POSITIVE" if pos >= 0.5 else "NEGATIVE"), pos

In [4]:
sample_reviews = [
    "This coffee is absolutely amazing! Best I have ever had.",
    "Terrible taste. The package was stale and I want a refund.",
    "It's okay, not the best but fine for the price.",
    "I was skeptical at first, but it turned out to be a great purchase.",
    "Shipping was fast but the snacks arrived crushed and stale.",
]

rows = []
for review in sample_reviews:
    l_label, l_conf = predict_lstm(review)
    b_label, b_conf = predict_bert(review)
    rows.append({"Review": review,
                 "LSTM": l_label, "LSTM_pos_prob": round(l_conf, 3),
                 "DistilBERT": b_label, "DistilBERT_pos_prob": round(b_conf, 3)})

df_results = pd.DataFrame(rows)
df_results.to_csv(f"{RESULTS_DIR}/demo_predictions.csv", index=False)
df_results

,Review,LSTM,LSTM_pos_prob,DistilBERT,DistilBERT_pos_prob
0,This coffee is absolutely amazing! Best I have...,POSITIVE,0.995,POSITIVE,0.997
1,Terrible taste. The package was stale and I wa...,NEGATIVE,0.002,NEGATIVE,0.004
2,"It's okay, not the best but fine for the price.",POSITIVE,0.863,NEGATIVE,0.073
3,"I was skeptical at first, but it turned out to...",POSITIVE,0.889,POSITIVE,0.994
4,Shipping was fast but the snacks arrived crush...,NEGATIVE,0.059,NEGATIVE,0.005


In [5]:
for i in range(6):   # DistilBERT has 6 layers: 0 to 5
    tokens, att = get_attention_map(sample_reviews[0], bert_model, tokenizer, layer_num=i)
    plot_attention_heatmap(tokens, att, layer_num=i,
                           save_path=f"{RESULTS_DIR}/attention_map_layer_{i}.png")

 Heatmap saved to ../results/attention_map_layer_0.png
 Heatmap saved to ../results/attention_map_layer_1.png
 Heatmap saved to ../results/attention_map_layer_2.png
 Heatmap saved to ../results/attention_map_layer_3.png
 Heatmap saved to ../results/attention_map_layer_4.png
 Heatmap saved to ../results/attention_map_layer_5.png
